In [5]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API key: ")

In [7]:
import sqlite3
from pathlib import Path

import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

DB = Path("spacex_launches.db")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
con = sqlite3.connect(DB)
df = pd.read_sql_query("SELECT * FROM launches ORDER BY launch_date", con)
con.close()
df

In [ ]:
test_suite = [
    {
        "question": "How many launches were successful?",
        "expected_sql": "SELECT COUNT(*) FROM launches WHERE success = 1",
        "expected_answer": "14",
    },
    {
        "question": "What was the heaviest payload ever launched?",
        "expected_sql": "SELECT mission_name, payload_kg FROM launches ORDER BY payload_kg DESC LIMIT 1",
        "expected_answer": "Starlink",
    },
    {
        "question": "Which vehicle has the highest success rate?",
        "expected_sql": "SELECT vehicle, ... GROUP BY vehicle ORDER BY ... DESC LIMIT 1",
        "expected_answer": "Falcon Heavy",
    },
    {
        # The DIFFERENTIATING test case.
        # 'heavy launch' looks obvious - the LLM will likely match it to the literal
        # `vehicle = 'Falcon Heavy'` value it sees in the schema (giving 2).
        # But our business defines a 'heavy launch' as `payload_kg > 5000` regardless
        # of vehicle. The correct answer is 6. Without this term in schema.md the LLM
        # gets it confidently wrong.
        "question": "How many heavy launches have we flown?",
        "expected_sql": "SELECT COUNT(*) FROM launches WHERE payload_kg > 5000",
        "expected_answer": "6",
    },
    {
        # Another domain term the LLM cannot guess: 'civilian mission' = customer = 'Shift4'
        "question": "How many civilian missions have flown?",
        "expected_sql": "SELECT COUNT(*) FROM launches WHERE customer = 'Shift4'",
        "expected_answer": "2",
    },
]

pd.DataFrame(test_suite)

In [10]:
con = sqlite3.connect(DB)
ddl = con.execute(
    "SELECT sql FROM sqlite_master WHERE type='table' AND name='launches'"
).fetchone()[0]
con.close()

print(ddl)

CREATE TABLE launches (
    launch_id      INTEGER PRIMARY KEY,
    mission_name   TEXT NOT NULL,
    vehicle        TEXT NOT NULL,                              -- Falcon 1 / Falcon 9 / Falcon Heavy / Starship
    payload_type   TEXT NOT NULL,                              -- Test / Cargo / Crew / Satellite / Probe
    payload_kg     INTEGER NOT NULL,                           -- 0 = no orbital payload (e.g. test flights)
    destination    TEXT NOT NULL,                              -- LEO / ISS / L1 / Heliocentric / Suborbital / ...
    success        INTEGER NOT NULL CHECK (success IN (0, 1)), -- 0 = failure, 1 = success (boolean)
    launch_date    DATE NOT NULL,
    customer       TEXT NOT NULL                               -- NASA / SpaceX / Iridium / NOAA / DARPA / ...
)


In [11]:
# Prompt v1: DDL only ----------------------------------------------------------
prompt_v1 = f"""You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

Schema (DDL only):
{ddl}
"""

print("--- prompt_v1 ---")
print(prompt_v1)

--- prompt_v1 ---
You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

Schema (DDL only):
CREATE TABLE launches (
    launch_id      INTEGER PRIMARY KEY,
    mission_name   TEXT NOT NULL,
    vehicle        TEXT NOT NULL,                              -- Falcon 1 / Falcon 9 / Falcon Heavy / Starship
    payload_type   TEXT NOT NULL,                              -- Test / Cargo / Crew / Satellite / Probe
    payload_kg     INTEGER NOT NULL,                           -- 0 = no orbital payload (e.g. test flights)
    destination    TEXT NOT NULL,                              -- LEO / ISS / L1 / Heliocentric / Suborbital / ...
    success        INTEGER NOT NULL CHECK (success IN (0, 1)), -- 0 = failure, 1 = success (boolean)
    launch_date    DATE NOT NULL,
    customer       TEXT NOT NULL                               -- NASA / SpaceX / Iridium / NOAA / DARPA / ...
)



In [12]:
# Prompt v2: + column descriptions and conventions -----------------------------
column_descriptions = """
Column descriptions:
- launch_id     (INTEGER): primary key, not meaningful.
- mission_name  (TEXT):    human-readable name like 'Crew Dragon Demo-2'.
- vehicle       (TEXT):    rocket family. Allowed values:
                           'Falcon 1', 'Falcon 9', 'Falcon Heavy', 'Starship'.
- payload_type  (TEXT):    Allowed values:
                           'Test', 'Cargo', 'Crew', 'Satellite', 'Probe'.
- payload_kg    (INTEGER): mass of payload in kg.
                           **0 means the launch had NO orbital payload (test flight).**
- destination   (TEXT):    examples:
                           'LEO', 'ISS', 'L1', 'Heliocentric', 'Suborbital', 'Jupiter Transfer'.
- success       (INTEGER): **1 = succeeded, 0 = failed.** Treat as boolean.
- launch_date   (DATE):    YYYY-MM-DD.
- customer      (TEXT):    examples:
                           'NASA', 'SpaceX', 'NOAA', 'Shift4', 'Iridium', 'MDA', 'DARPA'.

Conventions:
- Internal SpaceX flights have customer = 'SpaceX'.
- Test flights with no orbital payload have payload_kg = 0.
"""

prompt_v2 = f"""You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

Schema (DDL):
{ddl}

{column_descriptions}
"""

print("--- prompt_v2 (first 600 chars) ---")
print(prompt_v2[:600])
print("...")

--- prompt_v2 (first 600 chars) ---
You translate natural-language questions into SQLite SQL.

Use only the schema below. Return ONLY the SQL query.
No prose, no explanation, no markdown code fences.

Schema (DDL):
CREATE TABLE launches (
    launch_id      INTEGER PRIMARY KEY,
    mission_name   TEXT NOT NULL,
    vehicle        TEXT NOT NULL,                              -- Falcon 1 / Falcon 9 / Falcon Heavy / Starship
    payload_type   TEXT NOT NULL,                              -- Test / Cargo / Crew / Satellite / Probe
    payload_kg     INTEGER NOT NULL,                           -- 0 = no orbital payload (e.g. test fligh
...
